# Particle in a box

*quantum · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/infinite_well)

The time-independent Schrödinger equation on a grid becomes a matrix eigenvalue problem $H_{ij}\psi_j = E\psi_i$. torch.linalg.eigh gives the stationary states, and a superposition of them evolves by attaching a phase $e^{-iE_n t}$ to each one and summing with einsum.

**Vector form**

$$\hat H\,\psi = -\tfrac12\,\frac{d^2\psi}{dx^2} = E\,\psi,\qquad \psi(x,t) = \sum_n c_n\,\varphi_n(x)\,e^{-iE_n t}$$

**Index (tensor) form**

$$H_{xy}\,\varphi_{yn} = E_n\,\varphi_{xn},\qquad \psi_{x}(t) = \varphi_{xn}\,c_n\,e^{-iE_n t}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $H_{xy}$ | `hamiltonian(N, L)` | (N, N) | Hamiltonian matrix $-\tfrac12\,D2$ on the $N$ interior grid points (walls: $\psi = 0$) |
| $D2_{xy}$ | `second_derivative(N, dx)` | (N, N) | finite-difference $d^2/dx^2$: $(\psi_{x-1} - 2\psi_x + \psi_{x+1})/dx^2$ |
| $E_n,\ \varphi_{xn}$ | `torch.linalg.eigh(H)` | (N,), (N, N) | energies and eigenvectors (column $n$ is state $n$) |
| $c_n$ | `einsum('xn,x->n', phi, psi0) * dx` | (N,) | project the initial state onto the eigenbasis (repeated $x$ ⇒ summed) |
| $\psi_x(t)$ | `einsum('xn,n->x', phi, c * exp(-1j*E*t))` | (N,) | repeated $n$ ⇒ summed: rebuild $\psi$ from its phased components |

Units $\hbar = m = 1$. The exact energies are $E_n = n^2\pi^2/(2L^2)$; the grid reproduces the lowest ones to a small fraction of a percent. Each eigenstate only picks up a phase, so $\lvert c_n\rvert$ and the norm never change — the sloshing of $\lvert\psi\rvert^2$ is pure interference between states. After the revival time $T = 4L^2/\pi$ every phase $e^{-iE_n t}$ is back to 1 and the initial state reappears exactly. The faint dashed curves are the lowest stationary states $\varphi_n(x)$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/infinite_well/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import norm


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, L = params["n_grid"], params["L"]
    x, dx = tensors.grid(n, L, dev)
    H = tensors.hamiltonian(n, L, dev)

    energies, phi = solve.eigenstates(H, dx)
    if params["initial"] == "gaussian":
        c = solve.coefficients(phi, tensors.gaussian_packet(x, L, dx), dx)
        n_show = 3
    else:
        c = tensors.equal_superposition(params["n_states"], n, dev)
        n_show = max(3, params["n_states"])

    times, psi = solve.solve(phi, energies, c, params["revivals"] * solve.revival_time(L))

    for k, E in enumerate(energies[:4].tolist(), start=1):
        print(f"E_{k} = {E:9.4f}   exact n^2 pi^2/2L^2 = {(k * math.pi / L) ** 2 / 2:9.4f}")
    print(f"norm at t = 0: {norm(psi[0], dx):.6f}   at t_max: {norm(psi[-1], dx):.6f}")
    return plot.animate(x, times, psi, phi, n_show, L, out_dir / "infinite_well.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/infinite_well/equation.py
r"""Physics → tensor translation.

Time-independent Schrödinger equation for a particle in a box 0 < x < L (units ħ = m = 1):

    \hat H \varphi = -\tfrac12 \varphi'' = E \varphi,      \varphi(0) = \varphi(L) = 0.

On N grid points the operator becomes a matrix and the wavefunction a vector:

    H_{xy} \varphi_y = E \varphi_x,      H_{xy} = -\tfrac12 D2_{xy}      (y summed),

where D2 is the finite-difference second derivative. Collecting all N eigenvectors as the
columns of \varphi_{xn} (grid point x, state n) the whole eigenproblem is one line:

    H_{xy} \varphi_{yn} = E_n \varphi_{xn}.

Time evolution: each eigenstate only rotates its phase, so

    c_n = \varphi_{xn} \psi_x(0) \, dx                      (x summed: projection)
    \psi_x(t) = \varphi_{xn} \, c_n e^{-i E_n t}            (n summed: superposition)
"""

import torch


def project(phi: torch.Tensor, psi0: torch.Tensor, dx: float) -> torch.Tensor:
    r"""c_n = \varphi_{xn} \psi_x dx.   phi: (N, M), psi0: (N,) complex → (M,) complex"""
    return torch.einsum("xn,x->n", phi.to(psi0.dtype), psi0) * dx


def evolve(phi: torch.Tensor, energies: torch.Tensor, c: torch.Tensor, t: float) -> torch.Tensor:
    r"""\psi_x(t) = \varphi_{xn} c_n e^{-i E_n t}.   phi: (N, M), energies: (M,), c: (M,) → (N,) complex"""
    phase = torch.exp(-1j * energies * t)  # (M,)  e^{-i E_n t}
    return torch.einsum("xn,n->x", phi.to(c.dtype), c * phase)


def norm(psi: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\int |\psi|^2 dx \approx \psi^*_x \psi_x dx  (x summed).   (..., N) → (...)"""
    return torch.einsum("...x,...x->...", psi.conj(), psi).real * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/infinite_well/tensors.py
"""Build the concrete tensors: the grid, the Hamiltonian matrix and the initial wavefunction."""

import math

import torch


def grid(n: int, L: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """N interior points x_i = i·dx, i = 1..N, dx = L/(N+1). The walls x = 0, L are left out (ψ = 0 there)."""
    dx = L / (n + 1)
    x = dx * torch.arange(1, n + 1, dtype=torch.float64, device=device)  # (N,)
    return x, dx


def second_derivative(n: int, dx: float, device: torch.device) -> torch.Tensor:
    """D2_{xy}: −2 on the diagonal, 1 on the two neighbouring diagonals, all over dx². Shape (N, N)."""
    main = -2.0 * torch.ones(n, dtype=torch.float64, device=device)  # (N,)
    side = torch.ones(n - 1, dtype=torch.float64, device=device)  # (N-1,)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / dx**2


def hamiltonian(n: int, L: float, device: torch.device) -> torch.Tensor:
    """H_{xy} = −½ D2_{xy}  (kinetic energy only: V = 0 inside the box). Shape (N, N), symmetric."""
    _, dx = grid(n, L, device)
    return -0.5 * second_derivative(n, dx, device)


def equal_superposition(n_states: int, n: int, device: torch.device) -> torch.Tensor:
    """c_n = 1/√k for the lowest k states, 0 for the rest. Shape (N,) complex."""
    c = torch.zeros(n, dtype=torch.complex128, device=device)
    c[:n_states] = 1.0 / math.sqrt(n_states)
    return c


def gaussian_packet(x: torch.Tensor, L: float, dx: float) -> torch.Tensor:
    """ψ_x(0): a Gaussian at L/4 moving right with momentum k0 = 40/L, normalised so Σ|ψ|²dx = 1. Shape (N,)."""
    x0, sigma, k0 = 0.25 * L, 0.06 * L, 40.0 / L
    psi = torch.exp(-((x - x0) ** 2) / (4 * sigma**2) + 1j * k0 * x)  # (N,) complex128
    return psi / torch.sqrt((psi.abs() ** 2).sum() * dx)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/infinite_well/solve.py
"""Diagonalise H once, then evolve the superposition exactly in the eigenbasis."""

import math

import torch

from .equation import evolve, project


def eigenstates(H: torch.Tensor, dx: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (E_n (N,), φ_{xn} (N, N)) sorted by energy, normalised so Σ_x φ_{xn}² dx = 1.

    eigh is for symmetric (Hermitian) matrices: real eigenvalues, orthonormal eigenvectors.
    Its vectors have Σ_x φ² = 1; dividing by √dx turns that into the continuum normalisation ∫|φ|² dx = 1.
    """
    energies, phi = torch.linalg.eigh(H)
    return energies, phi / math.sqrt(dx)


def coefficients(phi: torch.Tensor, psi0: torch.Tensor, dx: float) -> torch.Tensor:
    """c_n of the initial state ψ0 in the eigenbasis: (N,) complex."""
    return project(phi, psi0, dx)


def revival_time(L: float) -> float:
    """T = 2π / E_1 = 4L²/π: every phase e^{−i n² E_1 T} is back to 1."""
    return 4 * L**2 / math.pi


def solve(
    phi: torch.Tensor, energies: torch.Tensor, c: torch.Tensor, t_max: float, n_steps: int = 240
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, N) complex). No time stepping: every t is evaluated exactly."""
    times = torch.linspace(0.0, t_max, n_steps + 1, dtype=torch.float64)
    psi = torch.stack([evolve(phi, energies, c, t) for t in times.tolist()])  # (T, N)
    return times, psi


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/infinite_well/plot.py
"""Animate |ψ|², Re ψ and Im ψ inside the box, with the lowest eigenstates drawn faintly behind."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def with_walls(f: torch.Tensor) -> torch.Tensor:
    """Append the boundary values ψ(0) = ψ(L) = 0 along the last axis: (..., N) → (..., N+2)."""
    zero = f.new_zeros(*f.shape[:-1], 1)
    return torch.cat([zero, f, zero], dim=-1)


def animate(
    x: torch.Tensor, times: torch.Tensor, psi: torch.Tensor, phi: torch.Tensor, n_show: int, L: float, path: Path
) -> Path:
    xs = torch.cat([x.new_zeros(1), x, x.new_full((1,), L)]).cpu().numpy()  # (N+2,) grid incl. walls
    psi = with_walls(psi).cpu()  # (T, N+2)
    prob = (psi.abs() ** 2).numpy()
    re, im = psi.real.numpy(), psi.imag.numpy()
    states = with_walls(phi[:, :n_show].T).cpu().numpy()  # (n_show, N+2)
    t = times.cpu().numpy()
    amp = float(psi.abs().max())

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(0, L)
    ax.set_ylim(-1.1 * amp, 1.1 * max(prob.max(), amp))
    ax.set_xlabel("x")
    ax.set_ylabel("ψ,  |ψ|²")
    for wall in (0, L):
        ax.axvline(wall, color="#374151", lw=3)
    for state in states:  # faint stationary states φ_n(x)
        ax.plot(xs, state, color="#9ca3af", lw=0.8, ls="--")

    (line_re,) = ax.plot([], [], color=ACCENT[0], lw=1.2, label="Re ψ")
    (line_im,) = ax.plot([], [], color=ACCENT[1], lw=1.2, label="Im ψ")
    (line_p,) = ax.plot([], [], color="#111827", lw=2.2, label="|ψ|²")
    fill = [ax.fill_between(xs, 0 * xs, color="#111827", alpha=0.08)]
    clock = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.legend(loc="upper right", fontsize=8)

    def update(step: int):
        line_re.set_data(xs, re[step])
        line_im.set_data(xs, im[step])
        line_p.set_data(xs, prob[step])
        fill[0].remove()
        fill[0] = ax.fill_between(xs, prob[step], color="#111827", alpha=0.08)
        clock.set_text(f"t = {t[step]:.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'initial': 'superposition',
    'n_states': 3,
    'L': 1.0,
    'n_grid': 300,
    'revivals': 1.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.infinite_well")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.infinite_well.main")
problem_dir = Path("problems/quantum/infinite_well")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
